# NIFTY 50 Market Crash Rebound Analysis

This notebook evaluates the historical performance and potential 'buy-the-dip' opportunities following significant daily drops (crashes) in the NIFTY 50 index over a 15-year period (2011–2026). We analyze short-term holding returns (1, 3, and 5 days) after significant market drops, compare them to market baselines, and test for statistical significance.

In [2]:
import yfinance as yf
import pandas as pd

# The NIFTY 50 ticker symbol on Yahoo Finance
ticker = "^NSEI"

# Fetch 15 years of daily market data
print("Downloading NIFTY 50 data...")
nifty_data = yf.download(ticker, start="2011-01-01", end="2026-01-01")

# The assignment requires Date, Open, High, Low, Close
nifty_data = nifty_data[['Open', 'High', 'Low', 'Close']]

# Drop any days where the market was closed or data is missing
nifty_data = nifty_data.dropna()

# Display the first 5 rows to confirm it worked
print(nifty_data.head())

/tmp/ipykernel_1648/2397826972.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  nifty_data = yf.download(ticker, start="2011-01-01", end="2026-01-01")
[*********************100%***********************]  1 of 1 completed

Price              Open         High          Low        Close
Ticker            ^NSEI        ^NSEI        ^NSEI        ^NSEI
Date                                                          
2011-01-03  6177.450195  6178.549805  6147.200195  6157.600098
2011-01-04  6172.750000  6181.049805  6124.399902  6146.350098
2011-01-05  6141.350098  6141.350098  6062.350098  6079.799805
2011-01-06  6107.000000  6116.149902  6022.299805  6048.250000
2011-01-07  6030.899902  6051.200195  5883.600098  5904.600098


## Event Definition & Identification

We define a market 'crash' event as a day where the close-to-close daily return is less than or equal to a designated negative threshold (initially -2.0%).

* -2.0% close-to-close daily drop

In [3]:
# The newer yfinance version gives us a double-header (Price/Ticker).
# This line cleans it up so our columns are just simple names.
nifty_data.columns = nifty_data.columns.get_level_values(0)

# Calculate the daily percentage return (Close compared to yesterday's Close)
nifty_data['Daily_Return'] = nifty_data['Close'].pct_change()

# Define our threshold for a "significant fall"
threshold = -0.02

# Flag the specific days where the daily return was worse than or equal to -2%
nifty_data['Event'] = nifty_data['Daily_Return'] <= threshold

# Count exactly how many times this happened in the last 15 years
event_count = nifty_data['Event'].sum()
print(f"Total -2% crash events found: {event_count}")

# Let's look at the first 5 times this happened to verify
print("\nFirst 5 Crash Events:")
display(nifty_data[nifty_data['Event'] == True].head())

Total -2% crash events found: 94

First 5 Crash Events:


Price,Open,High,Low,Close,Daily_Return,Event
Date,,,,,,
2011-01-07,6030.899902,6051.200195,5883.600098,5904.600098,-0.023751,True
2011-01-10,5901.299805,5907.250000,5740.950195,5762.850098,-0.024007,True
2011-02-04,5519.899902,5556.299805,5369.049805,5395.750000,-0.023703,True
2011-02-24,5408.750000,5423.399902,5242.500000,5262.700195,-0.032120,True
2011-05-03,5689.700195,5710.799805,5554.850098,5565.250000,-0.023863,True


## Post-Event Strategy & Returns Calculation

To simulate a realistic trading strategy, we assume an entry at the **Open price of the next trading day** following the crash. We calculate forward holding period returns over 1-day, 3-day, and 5-day windows relative to this entry price.

In [4]:
# 1. Next day open price
nifty_data['Next_Open'] = nifty_data['Open'].shift(-1)

# Find the holding period price
nifty_data['Exit_1D'] = nifty_data['Close'].shift(-1)
nifty_data['Exit_3D'] = nifty_data['Close'].shift(-3)
nifty_data['Exit_5D'] = nifty_data['Close'].shift(-5)

# 3. Returns calculate karenge: (Exit Price - Entry Price) / Entry Price
nifty_data['Return_1D'] = (nifty_data['Exit_1D'] - nifty_data['Next_Open']) / nifty_data['Next_Open']
nifty_data['Return_3D'] = (nifty_data['Exit_3D'] - nifty_data['Next_Open']) / nifty_data['Next_Open']
nifty_data['Return_5D'] = (nifty_data['Exit_5D'] - nifty_data['Next_Open']) / nifty_data['Next_Open']

# 4. Filter the data to only see those 94 crash event
events_df = nifty_data[nifty_data['Event'] == True].copy()

print("Returns after the first 5 crash events:")
display(events_df[['Next_Open', 'Return_1D', 'Return_3D', 'Return_5D']].head())

Returns after the first 5 crash events:


Price,Next_Open,Return_1D,Return_3D,Return_5D
Date,,,,
2011-01-07,5901.299805,-0.023461,-0.006448,-0.041813
2011-01-10,5767.950195,-0.002401,-0.002783,-0.019626
2011-02-04,5430.149902,-0.006289,-0.032522,-0.022126
2011-02-24,5321.049805,-0.003289,0.037821,0.040913
2011-05-03,5567.700195,-0.005487,-0.002919,-0.004751


## Performance Evaluation

Let's calculate key statistics (Mean Return, Median Return, and Win Rate) for each of the forward holding windows following our detected -2.0% crash events.

In [5]:
# 1. Total events
total_events = len(events_df)

# 2. Mean (Average), Median, and Win Rate
stats = pd.DataFrame({
    'Mean_Return (%)': events_df[['Return_1D', 'Return_3D', 'Return_5D']].mean() * 100,
    'Median_Return (%)': events_df[['Return_1D', 'Return_3D', 'Return_5D']].median() * 100,

    # Win Rate = Percentage of times the return is positive
    'Win_Rate (%)': (events_df[['Return_1D', 'Return_3D', 'Return_5D']] > 0).mean() * 100
})

print(f"Total -2% Crash Events: {total_events}\n")
print("Performance After Crash:")
display(stats.round(2))

Total -2% Crash Events: 94

Performance After Crash:


,Mean_Return (%),Median_Return (%),Win_Rate (%)
Price,,,
Return_1D,0.10,-0.09,46.81
Return_3D,0.23,0.55,58.51
Return_5D,0.12,0.70,53.19


## Baseline Performance Comparison

To determine if the performance following a crash is truly unique, we establish a baseline by calculating the mean, median, and win rate across all historical trading days in our dataset.

* Baseline Test

In [6]:
# Calculating baseline of all 15 years of data..
baseline_stats = pd.DataFrame({
    'Baseline_Mean (%)': nifty_data[['Return_1D', 'Return_3D', 'Return_5D']].mean() * 100,
    'Baseline_Median (%)': nifty_data[['Return_1D', 'Return_3D', 'Return_5D']].median() * 100,

    # Win rate of normal days
    'Baseline_Win_Rate (%)': (nifty_data[['Return_1D', 'Return_3D', 'Return_5D']] > 0).mean() * 100
})

print("Baseline: Normal Market Performance (All Days):")
display(baseline_stats.round(2))

Baseline: Normal Market Performance (All Days):


,Baseline_Mean (%),Baseline_Median (%),Baseline_Win_Rate (%)
Price,,,
Return_1D,-0.06,-0.05,46.67
Return_3D,0.02,0.10,52.70
Return_5D,0.12,0.20,54.12


## Statistical Significance Testing

We perform Welch's T-Test to evaluate whether the 3-day post-crash returns are statistically different from returns observed on non-crash (normal) trading days.

In [7]:
from scipy import stats

# 1. Separate the data: 3-day returns after a crash VS 3-day returns on normal days
# (We need to drop empty rows or NaN values to avoid test errors)
crash_returns = events_df['Return_3D'].dropna()

# Normal days (where there was no event)
normal_returns = nifty_data[nifty_data['Event'] == False]['Return_3D'].dropna()

# 2. Run Welch's T-Test (Since we have 94 crash days and about 3600 normal days)
t_stat, p_value = stats.ttest_ind(crash_returns, normal_returns, equal_var=False)

print("--- STATISTICAL TEST (3-Day Returns) ---")
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_value:.4f}\n")

# 3. Decision rule
if p_value < 0.05:
    print("Result: SIGNIFICANT! (This is not random chance; the pattern is real and differs from normal days.)")
else:
    print("Result: NOT SIGNIFICANT. (This bounce back could be due to random chance; the evidence is weak.)")

--- STATISTICAL TEST (3-Day Returns) ---
T-Statistic: 0.6468
P-Value: 0.5193

Result: NOT SIGNIFICANT. (Ye bounce back random chance ho sakta hai, evidence weak hai.)


## Parameter Robustness Check: -3% Threshold

We wrap our logic into a reusable testing engine to quickly evaluate different crash thresholds and check if a more severe drop (-3.0%) leads to a more pronounced or statistically significant rebound.

* What if we change the threshold to -3%

In [8]:
def test_crash_strategy(data, threshold):
    print(f"--- TESTING THRESHOLD: {threshold*100}% CRASH ---")

    # 1. Detect events
    data['Event'] = data['Daily_Return'] <= threshold
    events_df = data[data['Event'] == True].copy()

    total_events = len(events_df)
    print(f"Total Events Found: {total_events}")

    if total_events < 2:
        print("Not enough events to test.\n")
        return

    # 2. Stats calculate
    win_rate_3d = (events_df['Return_3D'] > 0).mean() * 100
    mean_3d = events_df['Return_3D'].mean() * 100

    print(f"3-Day Win Rate: {win_rate_3d:.2f}%")
    print(f"3-Day Average Return: {mean_3d:.2f}%")

    # 3. T-Test (Statistical Evidence)
    crash_returns = events_df['Return_3D'].dropna()
    normal_returns = data[data['Event'] == False]['Return_3D'].dropna()

    t_stat, p_value = stats.ttest_ind(crash_returns, normal_returns, equal_var=False)
    print(f"P-Value: {p_value:.4f}")

    if p_value < 0.05:
        print("Result: SIGNIFICANT! (Strong pattern)\n")
    else:
        print("Result: NOT SIGNIFICANT. (Weak evidence)\n")

# Ab hum is engine se koi bhi threshold ek second mein test kar sakte hain!
test_crash_strategy(nifty_data, -0.02)  # Purana -2% test
test_crash_strategy(nifty_data, -0.03)  # Naya -3% test

--- TESTING THRESHOLD: -2.0% CRASH ---
Total Events Found: 94
3-Day Win Rate: 58.51%
3-Day Average Return: 0.23%
P-Value: 0.5193
Result: NOT SIGNIFICANT. (Weak evidence)

--- TESTING THRESHOLD: -3.0% CRASH ---
Total Events Found: 25
3-Day Win Rate: 68.00%
3-Day Average Return: 0.76%
P-Value: 0.4116
Result: NOT SIGNIFICANT. (Weak evidence)



## Out-of-Sample Validation & Walk-Forward Check

To safeguard against overfitting, we divide our dataset chronologically into two periods:
* **In-Sample Period (2011 to 2021):** To analyze the initial pattern.
* **Out-of-Sample Period (2022 to 2026):** To validate if the pattern persists in unseen future data.

* dividing the data into two parts and threshold = -2%

In [9]:
# 1. Split the data chronologically into two periods
# Part 1: 2011 to 2021 (In-Sample / Research Period)
train_data = nifty_data.loc['2011-01-01':'2021-12-31'].copy()

# Part 2: 2022 to 2026 (Out-of-Sample / Unseen Validation Period)
test_data = nifty_data.loc['2022-01-01':'2026-01-01'].copy()

# 2. Run our -2% threshold engine on both datasets to compare performance
print("=== IN-SAMPLE (2011 to 2021) ===")
test_crash_strategy(train_data, -0.02)

print("=== OUT-OF-SAMPLE (2022 to 2026) ===")
test_crash_strategy(test_data, -0.02)

=== IN-SAMPLE (2011 to 2021) ===
--- TESTING THRESHOLD: -2.0% CRASH ---
Total Events Found: 80
3-Day Win Rate: 57.50%
3-Day Average Return: 0.11%
P-Value: 0.7958
Result: NOT SIGNIFICANT. (Weak evidence)

=== OUT-OF-SAMPLE (2022 to 2026) ===
--- TESTING THRESHOLD: -2.0% CRASH ---
Total Events Found: 14
3-Day Win Rate: 64.29%
3-Day Average Return: 0.95%
P-Value: 0.1884
Result: NOT SIGNIFICANT. (Weak evidence)

